# CupCat Bot Clone — verified Colab worker

This notebook starts the isolated bot-clone branch, brings up the FFmpeg worker and temporary tunnel, then runs a real end-to-end MP4 render test before the keep-alive cell.

- No paid AI API key is used.
- Ollama plus qwen2.5:3b is optional; if it cannot install, the worker can still test the FFmpeg rule-based rendering path.
- The notebook will clearly report worker health, local model health and render-test health separately. A tunnel HTTP response alone is not treated as proof that AI or rendering works.
- Keep this tab open while using the worker. The tunnel URL changes after a Colab restart.


In [ ]:
# 1/7 — helpers and isolated code branch
import os, re, sys, json, time, shutil, subprocess, threading, urllib.request, urllib.error
from pathlib import Path

REPO = "https://github.com/babyfam777-maker/capcutpro-agent"
BRANCH = "cursor/cupcat-bot-clone"
ROOT = "/content/capcutpro-agent"
WORKER_DIR = ROOT + "/worker"
PORT = 8787
DATA_DIR = "/content/data"
MEMORY_FILE = DATA_DIR + "/cupcat_memory_overrides.json"
os.makedirs(DATA_DIR, exist_ok=True)

def log(*items):
    print(*items, flush=True)

def sh(cmd, timeout=120, name="command", allow_failure=False):
    log(f"[{name}] running")
    try:
        result = subprocess.run(cmd, shell=True, timeout=timeout, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
    except subprocess.TimeoutExpired:
        log(f"[{name}] TIMEOUT after {timeout}s")
        if allow_failure:
            return False
        raise
    if result.stdout:
        log(result.stdout[-3500:])
    if result.returncode:
        log(f"[{name}] FAILED exit={result.returncode}")
        if allow_failure:
            return False
        raise RuntimeError(f"{name} failed with exit {result.returncode}")
    log(f"[{name}] OK")
    return True

def get_health(url, timeout=5):
    try:
        req = urllib.request.Request(url.rstrip("/") + "/health", headers={"Cache-Control": "no-cache"})
        with urllib.request.urlopen(req, timeout=timeout) as response:
            body = response.read().decode("utf-8", "replace")
            payload = json.loads(body)
            return response.status == 200 and bool(payload.get("ffmpeg")), payload
    except Exception as exc:
        return False, {"error": str(exc)}

def wait_health(url, timeout=120):
    deadline = time.time() + timeout
    last = {}
    while time.time() < deadline:
        ok, last = get_health(url, timeout=4)
        if ok:
            return True, last
        time.sleep(2)
    return False, last

sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} {ROOT}", timeout=180, name="clone bot-clone branch")
log("BRANCH:", BRANCH)
log("WORKER FILES:", ", ".join(sorted(os.listdir(WORKER_DIR))))


In [ ]:
# 2/7 — system packages and Python dependencies
sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ffmpeg fonts-noto-core fonts-noto-color-emoji curl zstd libegl1 libgles2 libgl1", timeout=600, name="system packages")
sh(f"grep -v '^pytest' {WORKER_DIR}/requirements.txt > /tmp/cupcat-req.txt && pip -q install -r /tmp/cupcat-req.txt", timeout=900, name="Python dependencies")
sh("ffmpeg -version | head -n 1 && python -c 'import fastapi, cv2, yaml; print(\"Python imports OK\")'", timeout=90, name="dependency smoke test")
log("DEPENDENCIES READY")


In [ ]:
# 3/7 — install local AI if possible (not required for the render smoke test)
MODEL_OK = False
ollama_proc = None

try:
    if not shutil.which("ollama"):
        sh("curl -fsSL https://ollama.com/install.sh | sh", timeout=360, name="Ollama install")
    sh("ollama --version", timeout=30, name="Ollama version")

    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
        log("Ollama server already running")
    except Exception:
        ollama_proc = subprocess.Popen(
            ["ollama", "serve"],
            stdout=open("/content/ollama.log", "a"),
            stderr=subprocess.STDOUT
        )
        for _ in range(60):
            try:
                urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see /content/ollama.log")

    MODEL_OK = sh("ollama pull qwen2.5:3b", timeout=900, name="download qwen2.5:3b", allow_failure=True)
except Exception as exc:
    log("LOCAL MODEL INSTALL FAILED:", repr(exc))
    log("The worker will still start; rule-based editing and MP4 rendering can be tested.")

log("MODEL READY: qwen2.5:3b" if MODEL_OK else "MODEL UNAVAILABLE: continuing without the local LLM")


In [ ]:
# 4/7 — start worker and report real health fields
WLOG = "/content/worker.log"
ENV = dict(
    os.environ,
    DATA_DIR=DATA_DIR,
    CUPCAT_MEMORY_FILE=MEMORY_FILE,
    WHISPER_MODEL="tiny",
    OPENAI_BASE_URL="http://127.0.0.1:11434/v1",
    OPENAI_API_KEY="ollama",
    MODEL="qwen2.5:3b",
    SHORT_DELIVERY="phone",
    WORKER_TOKEN="",
    PYTHONUNBUFFERED="1",
)
worker = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=WORKER_DIR,
    env=ENV,
    stdout=open(WLOG, "a"),
    stderr=subprocess.STDOUT,
)
worker_ok, worker_health = wait_health(f"http://127.0.0.1:{PORT}", timeout=150)
if not worker_ok:
    log("WORKER START FAILED. LOG TAIL:")
    log("".join(Path(WLOG).read_text(errors="replace").splitlines(True)[-80:]))
    raise RuntimeError("Worker /health did not report FFmpeg ready.")

log("WORKER / FFMPEG: READY")
log("LLM STATUS:", json.dumps(worker_health.get("llm", {}), ensure_ascii=False))
if not (worker_health.get("llm") or {}).get("ok"):
    log("NOTE: local AI model is not ready. The worker may use its explicitly labeled rule-based fallback.")


In [ ]:
# 5/7 — start temporary tunnel and check worker health through it
CF = "/content/cloudflared"
TLOG = "/content/tunnel.log"
if not os.path.exists(CF):
    sh(f"curl -fsSL -o {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}",
       timeout=150, name="download cloudflared")

def start_tunnel(wait_url=75, wait_health=90):
    Path(TLOG).write_text("")
    proc = subprocess.Popen(
        [CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=open(TLOG, "a"),
        stderr=subprocess.STDOUT,
    )
    url = None
    deadline = time.time() + wait_url
    while time.time() < deadline and proc.poll() is None:
        try:
            raw = Path(TLOG).read_text(errors="replace")
        except Exception:
            raw = ""
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", raw)
        if match:
            url = match.group(0)
            break
        time.sleep(1)
    if not url:
        log("TUNNEL FAILED. LOG TAIL:")
        log("".join(Path(TLOG).read_text(errors="replace").splitlines(True)[-60:]))
        if proc.poll() is None:
            proc.kill()
        return None, None
    ok, payload = wait_health(url, timeout=wait_health)
    if not ok:
        log("Tunnel URL exists but /health did not report FFmpeg ready:", json.dumps(payload, ensure_ascii=False))
        if proc.poll() is None:
            proc.kill()
        return None, None
    log("TUNNEL HTTP: READY")
    log("TUNNEL FFMPEG: READY")
    llm = payload.get("llm") or {}
    log("TUNNEL LOCAL AI:", "READY" if llm.get("ok") else "UNAVAILABLE / RULE-BASED FALLBACK")
    log("WORKER URL:", url)
    return proc, url

tunnel, URL = start_tunnel()
if not URL:
    raise RuntimeError("Could not establish a verified worker tunnel. Check the tunnel log above.")


In [ ]:
# 6/7 — REAL end-to-end test: synthetic input -> worker -> rendered MP4
# This tests the FFmpeg editing path even if the local LLM could not be installed.
import requests

TEST_INPUT = "/content/cupcat_e2e_input.mp4"
TEST_OUTPUT = "/content/CupCat_TEST_RENDER.mp4"
make_test = (
    'ffmpeg -hide_banner -loglevel error -y '
    '-f lavfi -i "testsrc2=size=720x1280:rate=24:duration=6" '
    '-f lavfi -i "sine=frequency=440:sample_rate=48000:duration=6" '
    '-shortest -c:v libx264 -preset ultrafast -pix_fmt yuv420p -c:a aac -b:a 96k '
    + TEST_INPUT
)
sh(make_test, timeout=120, name="create synthetic test clip")

with open(TEST_INPUT, "rb") as video_file:
    upload = requests.post(
        f"http://127.0.0.1:{PORT}/projects",
        files={"file": ("cupcat-e2e-test.mp4", video_file, "video/mp4")},
        timeout=(10, 120),
    )
upload.raise_for_status()
project = upload.json()
project_id = project.get("id")
if not project_id:
    raise RuntimeError(f"Worker did not return a project id: {project}")

log("TEST PROJECT CREATED:", project_id)
payload = {"prompt": "Gör en kort vertikal YouTube Short av testklippet. Behåll originalljudet och lägg inte till musik."}
with requests.post(
    f"http://127.0.0.1:{PORT}/projects/{project_id}/short",
    json=payload,
    stream=True,
    timeout=(10, 600),
) as response:
    response.raise_for_status()
    current_event = None
    data_lines = []
    done_data = None
    error_data = None
    for raw_line in response.iter_lines(decode_unicode=True):
        line = raw_line or ""
        if line.startswith("event: "):
            current_event = line[len("event: "):].strip()
        elif line.startswith("data: "):
            data_lines.append(line[len("data: "):])
        elif not line:
            if current_event and data_lines:
                joined = "\n".join(data_lines)
                try:
                    event_data = json.loads(joined)
                except Exception:
                    event_data = {"raw": joined}
                if current_event == "status":
                    log("RENDER STATUS:", event_data.get("step"))
                elif current_event == "progress":
                    log("RENDER PROGRESS:", event_data.get("progress"))
                elif current_event == "error":
                    error_data = event_data
                    log("RENDER ERROR:", json.dumps(event_data, ensure_ascii=False))
                    break
                elif current_event == "done":
                    done_data = event_data
                    log("RENDER DONE:", json.dumps(event_data, ensure_ascii=False)[:1600])
                    break
            current_event = None
            data_lines = []
    if error_data:
        raise RuntimeError("End-to-end render failed: " + str(error_data))
    if not done_data:
        raise RuntimeError("Worker returned no final done event; check /content/worker.log")

version = done_data.get("activeVersion")
if not version:
    raise RuntimeError("The worker returned done but no active MP4 version.")
download = requests.get(
    f"http://127.0.0.1:{PORT}/projects/{project_id}/versions/{version}/video?download=1",
    timeout=(10, 300),
)
download.raise_for_status()
Path(TEST_OUTPUT).write_bytes(download.content)
if not os.path.exists(TEST_OUTPUT) or os.path.getsize(TEST_OUTPUT) < 10000:
    raise RuntimeError("Render output is missing or suspiciously small.")

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_name,width,height",
     "-of", "json", TEST_OUTPUT],
    capture_output=True, text=True, timeout=30,
)
if probe.returncode != 0:
    raise RuntimeError("Output MP4 did not pass ffprobe: " + probe.stderr[-1000:])
details = json.loads(probe.stdout)
video_stream = next((s for s in details.get("streams", []) if s.get("codec_name") is not None and "width" in s), {})
duration = float((details.get("format") or {}).get("duration") or 0)
log("E2E RENDER PASS")
log("OUTPUT:", TEST_OUTPUT)
log("SIZE_MB:", round(os.path.getsize(TEST_OUTPUT) / 1048576, 2))
log("DURATION_SEC:", round(duration, 2))
log("VIDEO_SIZE:", f"{video_stream.get('width')}x{video_stream.get('height')}")
log("MODEL_USED:", "Ollama qwen2.5:3b" if MODEL_OK else "rule-based fallback; no model ready")


In [ ]:
# 7/7 — keep worker and tunnel alive; stop this cell when finished
while True:
    if worker.poll() is not None:
        log("WORKER STOPPED. Restarting...")
        worker = subprocess.Popen(
            [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
            cwd=WORKER_DIR, env=ENV,
            stdout=open(WLOG, "a"), stderr=subprocess.STDOUT,
        )
        time.sleep(5)
    if tunnel.poll() is not None:
        log("TUNNEL STOPPED. Restarting...")
        tunnel, URL = start_tunnel()
    ok, state = get_health(f"http://127.0.0.1:{PORT}", timeout=4)
    log(time.strftime("%H:%M:%S"), "| worker", "up" if ok else "DOWN",
        "| local AI", "up" if (state.get("llm") or {}).get("ok") else "fallback",
        "| tunnel", "up" if tunnel.poll() is None else "DOWN", "|", URL, flush=True)
    time.sleep(60)
